In [ ]:
import hashlib
import hmac
import math
import os
import sqlite3
import tempfile
import threading
from datetime import datetime, timedelta

# ------------------------------------------------------------------ CONFIG
EXPERIMENT_ID = "EXP-ONB-RECOVERY-001"
SECRET = b"demo-secret-change-me"       # that su: secret manager, KHONG commit len git
ALGORITHM_VERSION = "hmac-sha256-v2"
BUCKETS = 10_000
DEFAULT_TREATMENT_BP = 5_000            # 50/50 (bp = 1/10000). 90/10 -> 9000. Dong bang khi bat dau thi nghiem.

# Ly do chan (suppression reason). Chi la du lieu ghi them, khong sua/xoa.
REASONS = (
    "OPT_OUT",            # khach tu choi nhan tin (REC 05, Nghi dinh 91/2020)
    "COMPLAINT",          # khieu nai
    "COMPLETED",          # da mo tai khoan / hoan thanh ho so (thay cho "da mua hang")
    "DO_NOT_CONTACT",     # yeu cau cua ngan hang / phap ly
    "REVIEW_PENDING",     # so dien thoai dung chung, cho ra soat (EVT 05)
)

DDL = """
CREATE TABLE IF NOT EXISTS experiment_assignment (
    assignment_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    experiment_id        TEXT NOT NULL,
    applicant_key        TEXT NOT NULL,
    arm                  TEXT NOT NULL CHECK (arm IN ('TREATMENT','CONTROL')),
    index_application_id TEXT NOT NULL,
    hash_bucket          INTEGER NOT NULL,
    treatment_share_bp   INTEGER NOT NULL,
    assigned_at          TEXT NOT NULL,
    algorithm_version    TEXT NOT NULL,
    UNIQUE (experiment_id, applicant_key)
);
CREATE TABLE IF NOT EXISTS suppression_list (
    suppression_id INTEGER PRIMARY KEY AUTOINCREMENT,
    applicant_key  TEXT NOT NULL,
    reason         TEXT NOT NULL CHECK (reason IN ('OPT_OUT','COMPLAINT','COMPLETED','DO_NOT_CONTACT','REVIEW_PENDING')),
    source         TEXT NOT NULL,          -- he thong nguon: sms_stop, bank_core, staff, ...
    effective_at   TEXT NOT NULL,          -- co hieu luc tu thoi diem nay
    created_at     TEXT NOT NULL,
    UNIQUE (applicant_key, reason)
);
CREATE TABLE IF NOT EXISTS dispatch_decision_log (
    log_id        INTEGER PRIMARY KEY AUTOINCREMENT,
    applicant_key TEXT NOT NULL,
    arm           TEXT,
    allowed       INTEGER NOT NULL,
    reason        TEXT NOT NULL,
    decided_at    TEXT NOT NULL
);
"""


def connect(path: str) -> sqlite3.Connection:
    conn = sqlite3.connect(path, timeout=30, check_same_thread=False)
    conn.executescript(DDL)
    return conn


def _now_iso(now=None):
    return (now or datetime.now()).isoformat(timespec="seconds")


# ============================================================ 1. HOLDOUT GROUP
def hash_bucket(applicant_key: str, experiment_id: str = EXPERIMENT_ID, secret: bytes = SECRET) -> int:
    msg = f"{experiment_id}|{applicant_key}".encode()
    digest = hmac.new(secret, msg, hashlib.sha256).digest()
    return int.from_bytes(digest[:8], "big") % BUCKETS


def arm_from_bucket(bucket: int, treatment_bp: int = DEFAULT_TREATMENT_BP) -> str:
    """bucket < treatment_bp -> TREATMENT (nhan tin); con lai -> CONTROL (holdout, khong nhan tin)."""
    return "TREATMENT" if bucket < treatment_bp else "CONTROL"


def assign(conn, applicant_key, application_id, eligible, now=None, treatment_bp=DEFAULT_TREATMENT_BP):
    """
    Chia nhom mot lan duy nhat cho moi nguoi.
    Tra ve dict hoac None (khong du dieu kien / dang bi chan truoc khi chia).
    - Nguoi da nam trong suppression list tai thoi diem chia nhom -> khong chia (loai ra khoi thi nghiem,
      doi xung cho ca hai nhom vi quyet dinh nay chua dung toi nhom).
    - Da co dong dang ky -> tra lai dong cu (nhom, index application khong doi), bo qua treatment_bp moi.
    """
    if not eligible:
        return None
    now = now or datetime.now()
    existing = _get_assignment(conn, applicant_key)
    if existing is None and active_suppressions(conn, applicant_key, now):
        return None
    b = hash_bucket(applicant_key)
    conn.execute(
        """INSERT OR IGNORE INTO experiment_assignment
           (experiment_id, applicant_key, arm, index_application_id, hash_bucket,
            treatment_share_bp, assigned_at, algorithm_version)
           VALUES (?,?,?,?,?,?,?,?)""",
        (EXPERIMENT_ID, applicant_key, arm_from_bucket(b, treatment_bp), application_id, b,
         treatment_bp, _now_iso(now), ALGORITHM_VERSION),
    )
    conn.commit()
    return _get_assignment(conn, applicant_key)


def _get_assignment(conn, applicant_key):
    row = conn.execute(
        """SELECT arm, index_application_id, hash_bucket, treatment_share_bp, assigned_at
           FROM experiment_assignment WHERE experiment_id=? AND applicant_key=?""",
        (EXPERIMENT_ID, applicant_key),
    ).fetchone()
    if row is None:
        return None
    return dict(arm=row[0], index_application_id=row[1], hash_bucket=row[2],
                treatment_share_bp=row[3], assigned_at=row[4])


# ======================================================== 2. SUPPRESSION LIST
def add_suppression(conn, applicant_key, reason, source, effective_at=None, now=None):
    """
    Them mot ly do chan. Ghi them, khong sua/xoa; goi lap lai khong tao dong trung (UNIQUE).
    Tra ve True neu la dong moi, False neu da ton tai.
    """
    if reason not in REASONS:
        raise ValueError(f"reason khong hop le: {reason}")
    now = now or datetime.now()
    cur = conn.execute(
        """INSERT OR IGNORE INTO suppression_list
           (applicant_key, reason, source, effective_at, created_at) VALUES (?,?,?,?,?)""",
        (applicant_key, reason, source, _now_iso(effective_at or now), _now_iso(now)),
    )
    conn.commit()
    return cur.rowcount == 1


def active_suppressions(conn, applicant_key, at=None):
    """Danh sach ly do chan dang co hieu luc tai thoi diem `at`.=="""
    at = at or datetime.now()
    rows = conn.execute(
        "SELECT reason FROM suppression_list WHERE applicant_key=? AND effective_at<=?",
        (applicant_key, _now_iso(at)),
    ).fetchall()
    return [r[0] for r in rows]


# ==================================================== 3. CONG DISPATCH (BACKEND)
def check_before_send(conn, applicant_key, now=None):
    """
    Ham Backend goi NGAY TRUOC moi lan gui nhac nho thu nghiem (REC 06).
    Tra ve (allowed: bool, reason: str). Dong vai tro MOT trong cac cong; Treatment van phai qua cac cong khac
    (trang thai ho so, quyen lien he, gio yen lang, ngan sach, phien ban chien dich).

    Thu tu kiem tra:
      1. Khong co dong dang ky      -> chan (chua chia nhom)
      2. arm = CONTROL (holdout)    -> chan
      3. Co ly do chan hieu luc     -> chan (kem ly do)
      4. Con lai                    -> cho di tiep
    Loi bat ky (DB loi, du lieu hong) -> chan (fail closed), khong bao gio mac dinh la cho gui.
    """
    now = now or datetime.now()
    arm = None
    try:
        a = _get_assignment(conn, applicant_key)
        if a is None:
            allowed, reason = False, "BLOCK_NOT_ASSIGNED"
        else:
            arm = a["arm"]
            if arm != "TREATMENT":
                allowed, reason = False, "BLOCK_CONTROL_HOLDOUT"
            else:
                sup = active_suppressions(conn, applicant_key, now)
                if sup:
                    allowed, reason = False, "BLOCK_SUPPRESSED:" + ",".join(sorted(sup))
                else:
                    allowed, reason = True, "ALLOW_PASS_TO_NEXT_GATES"
    except Exception as exc:                      # fail closed
        return False, f"BLOCK_ERROR:{type(exc).__name__}"
    try:
        conn.execute(
            "INSERT INTO dispatch_decision_log (applicant_key, arm, allowed, reason, decided_at) VALUES (?,?,?,?,?)",
            (applicant_key, arm, int(allowed), reason, _now_iso(now)),
        )
        conn.commit()
    except Exception:
        pass                                      # loi ghi log khong duoc lam doi quyet dinh chan
    return allowed, reason


# ------------------------------------------------------------------ TESTS
def chi2_p_value(chi2):
    return math.erfc(math.sqrt(chi2 / 2))


def _fresh():
    return connect(os.path.join(tempfile.mkdtemp(), "demo.db"))


def run_tests():
    t0 = datetime(2026, 11, 12, 9, 0, 0)
    results = []

    def check(name, cond, detail=""):
        print(f"TEST {len(results)+1} - {name}\n  {'PASS' if cond else 'FAIL'} {detail}")
        results.append(cond)

    # 1. Holdout 50/50 va 90/10 dung ty le (kiem dinh chi binh phuong, p >= 0,001)
    for bp, label in ((5000, "50/50"), (9000, "90/10")):
        n = 100_000
        t = sum(arm_from_bucket(hash_bucket(f"APL-{i:06d}"), bp) == "TREATMENT" for i in range(n))
        exp = n * bp / BUCKETS
        chi2 = (t - exp) ** 2 / exp + ((n - t) - (n - exp)) ** 2 / (n - exp)
        p = chi2_p_value(chi2)
        check(f"Ty le chia {label} tren 100.000 nguoi", p >= 0.001, f"(Treatment {t}, ky vong {exp:.0f}, p = {p:.3f})")

    # 2. Doi ty le sau khi da chia: nguoi cu giu nhom cu
    conn = _fresh()
    a1 = assign(conn, "APL-1", "APP-1", True, t0, treatment_bp=5000)
    a2 = assign(conn, "APL-1", "APP-2", True, t0 + timedelta(days=1), treatment_bp=9000)
    n_rows = conn.execute("SELECT COUNT(*) FROM experiment_assignment").fetchone()[0]
    check("Doi ty le giua chung: nguoi da chia giu nguyen nhom", a1 == a2 and n_rows == 1)

    # 3. Khong chia nhom khi khong du dieu kien
    conn = _fresh()
    check("Khong du dieu kien -> khong chia nhom", assign(conn, "APL-2", "APP-1", False, t0) is None)

    # 4. Da co ly do chan truoc khi chia -> khong chia
    conn = _fresh()
    add_suppression(conn, "APL-3", "COMPLETED", "bank_core", now=t0 - timedelta(days=1))
    check("Da hoan thanh ho so truoc khi chia -> khong chia nhom", assign(conn, "APL-3", "APP-1", True, t0) is None)

    # Tim hai nguoi thuoc Treatment va Control (theo 50/50) de test cong
    keys = [f"APL-T{i}" for i in range(200)]
    tkey = next(k for k in keys if arm_from_bucket(hash_bucket(k)) == "TREATMENT")
    ckey = next(k for k in keys if arm_from_bucket(hash_bucket(k)) == "CONTROL")

    # 5. Cong: Treatment sach -> cho di tiep; Control -> chan; chua chia -> chan
    conn = _fresh()
    assign(conn, tkey, "APP-1", True, t0)
    assign(conn, ckey, "APP-1", True, t0)
    ok_t = check_before_send(conn, tkey, t0)
    ok_c = check_before_send(conn, ckey, t0)
    ok_n = check_before_send(conn, "APL-CHUA-CHIA", t0)
    check("Cong dispatch: Treatment cho qua, Control va chua chia bi chan",
          ok_t == (True, "ALLOW_PASS_TO_NEXT_GATES") and ok_c == (False, "BLOCK_CONTROL_HOLDOUT")
          and ok_n == (False, "BLOCK_NOT_ASSIGNED"))

    # 6. Opt-out SAU khi chia: van chan gui, nhung giu nhom (ITT)
    conn = _fresh()
    assign(conn, tkey, "APP-1", True, t0)
    add_suppression(conn, tkey, "OPT_OUT", "sms_stop", now=t0 + timedelta(days=2))
    allowed, reason = check_before_send(conn, tkey, t0 + timedelta(days=3))
    still = _get_assignment(conn, tkey)
    check("Opt-out sau khi chia: chan gui, giu nguyen nhom (ITT)",
          not allowed and reason == "BLOCK_SUPPRESSED:OPT_OUT" and still["arm"] == "TREATMENT")

    # 7. Them ly do chan giua luc chia nhom va luc gui -> cong phai thay
    conn = _fresh()
    assign(conn, tkey, "APP-1", True, t0)
    before = check_before_send(conn, tkey, t0 + timedelta(minutes=1))
    add_suppression(conn, tkey, "COMPLAINT", "staff", now=t0 + timedelta(minutes=2))
    after = check_before_send(conn, tkey, t0 + timedelta(minutes=3))
    check("Kiem tra tai thoi diem gui (khong dung ket qua cu)", before[0] and not after[0])

    # 8. Ly do chan co hieu luc trong tuong lai chua chan
    conn = _fresh()
    assign(conn, tkey, "APP-1", True, t0)
    add_suppression(conn, tkey, "DO_NOT_CONTACT", "bank", effective_at=t0 + timedelta(days=5), now=t0)
    now_ok = check_before_send(conn, tkey, t0 + timedelta(days=1))[0]
    later = check_before_send(conn, tkey, t0 + timedelta(days=6))[0]
    check("effective_at trong tuong lai: chua chan, den han thi chan", now_ok and not later)

    # 9. Goi lap them cung ly do khong tao dong trung; ly do la la bi tu choi
    conn = _fresh()
    first = add_suppression(conn, "APL-9", "OPT_OUT", "sms_stop", now=t0)
    second = add_suppression(conn, "APL-9", "OPT_OUT", "sms_stop", now=t0 + timedelta(hours=1))
    rows = conn.execute("SELECT COUNT(*) FROM suppression_list").fetchone()[0]
    try:
        add_suppression(conn, "APL-9", "BUY_ALREADY", "x")
        bad = False
    except ValueError:
        bad = True
    check("Them lap khong trung dong; ly do la bi tu choi", first and not second and rows == 1 and bad)

    # 10. Nhieu ly do cung luc deu duoc ghi trong ly do chan
    conn = _fresh()
    assign(conn, tkey, "APP-1", True, t0)
    add_suppression(conn, tkey, "OPT_OUT", "sms_stop", now=t0)
    add_suppression(conn, tkey, "COMPLETED", "bank_core", now=t0)
    allowed, reason = check_before_send(conn, tkey, t0 + timedelta(hours=1))
    check("Nhieu ly do chan: tra ve du ca hai", (not allowed) and reason == "BLOCK_SUPPRESSED:COMPLETED,OPT_OUT")

    # 11. Fail closed: loi DB -> chan, khong bao gio cho gui
    conn = _fresh()
    assign(conn, tkey, "APP-1", True, t0)
    conn.close()
    allowed, reason = check_before_send(conn, tkey, t0)
    check("Loi co so du lieu -> chan (fail closed)", (not allowed) and reason.startswith("BLOCK_ERROR"))

    # 12. 50 luong dong thoi chia cung mot nguoi -> 1 dong, 1 nhom
    path = os.path.join(tempfile.mkdtemp(), "conc.db")
    connect(path).close()
    out = []

    def worker():
        c = connect(path)
        out.append(assign(c, "APL-CONC", "APP-1", True, t0)["arm"])
        c.close()

    threads = [threading.Thread(target=worker) for _ in range(50)]
    [t.start() for t in threads]
    [t.join() for t in threads]
    rows = connect(path).execute("SELECT COUNT(*) FROM experiment_assignment").fetchone()[0]
    check("50 luong dong thoi: 1 dong, 1 nhom", rows == 1 and len(set(out)) == 1)

    # 13. Moi quyet dinh cong deu duoc ghi log (DAT 03, SEC 04)
    conn = _fresh()
    assign(conn, tkey, "APP-1", True, t0)
    assign(conn, ckey, "APP-1", True, t0)
    check_before_send(conn, tkey, t0)
    check_before_send(conn, ckey, t0)
    n_log = conn.execute("SELECT COUNT(*) FROM dispatch_decision_log").fetchone()[0]
    check("Moi quyet dinh cong duoc ghi log", n_log == 2)

    print("\n=> TAT CA TEST DAT" if all(results) else "\n=> CO TEST THAT BAI")
    return all(results)


if __name__ == "__main__":
    # Do not raise SystemExit inside an interactive Jupyter/Colab environment
    run_tests()

TEST 1 - Ty le chia 50/50 tren 100.000 nguoi
  PASS (Treatment 50149, ky vong 50000, p = 0.346)
TEST 2 - Ty le chia 90/10 tren 100.000 nguoi
  PASS (Treatment 90051, ky vong 90000, p = 0.591)
TEST 3 - Doi ty le giua chung: nguoi da chia giu nguyen nhom
  PASS 
TEST 4 - Khong du dieu kien -> khong chia nhom
  PASS 
TEST 5 - Da hoan thanh ho so truoc khi chia -> khong chia nhom
  PASS 
TEST 6 - Cong dispatch: Treatment cho qua, Control va chua chia bi chan
  PASS 
TEST 7 - Opt-out sau khi chia: chan gui, giu nguyen nhom (ITT)
  PASS 
TEST 8 - Kiem tra tai thoi diem gui (khong dung ket qua cu)
  PASS 
TEST 9 - effective_at trong tuong lai: chua chan, den han thi chan
  PASS 
TEST 10 - Them lap khong trung dong; ly do la bi tu choi
  PASS 
TEST 11 - Nhieu ly do chan: tra ve du ca hai
  PASS 
TEST 12 - Loi co so du lieu -> chan (fail closed)
  PASS 
TEST 13 - 50 luong dong thoi: 1 dong, 1 nhom
  PASS 
TEST 14 - Moi quyet dinh cong duoc ghi log
  PASS 

=> TAT CA TEST DAT
